In [15]:
import torch
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris, fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [16]:
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

def softmax(z):
    z = z - z.max(dim=1, keepdim=True)[0]
    return torch.exp(z) / torch.exp(z).sum(dim=1, keepdim=True)

In [17]:
def bce(p, y):
    return -(y * torch.log(p) + (1 - y) * torch.log(1 - p)).mean()

def cce(p, y):
    return -(y * torch.log(p)).sum(dim=1).mean()

def mse(y_hat, y):
    return ((y_hat - y) ** 2).mean()

def mae(y_hat, y):
    return torch.abs(y_hat - y).mean()

In [18]:
def forward(X, W1, b1, W2, b2, out="sigmoid"):
    h = sigmoid(X @ W1 + b1)
    z = h @ W2 + b2
    if out == "sigmoid":
        return sigmoid(z)
    if out == "softmax":
        return softmax(z)
    return z

In [19]:
df = pd.read_csv("pumpkin_seeds.csv", encoding="latin1")
X = df.iloc[:, :-1].values
y_raw = df.iloc[:, -1].values
classes = np.unique(y_raw)
y = (y_raw == classes[1]).astype(np.float32).reshape(-1, 1)

X = torch.tensor(StandardScaler().fit_transform(X), dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32)

D, H = X.shape[1], 16
W1 = torch.randn(D, H) * 0.1
b1 = torch.zeros(H)
W2 = torch.randn(H, 1) * 0.1
b2 = torch.zeros(1)

p = forward(X, W1, b1, W2, b2, out="sigmoid")
print("Pumpkin BCE:", bce(p, y).item())

Pumpkin BCE: 0.6954908967018127


In [20]:
iris = load_iris()
X = iris.data
y = iris.target
C = len(np.unique(y))
y_oh = np.eye(C)[y]

X = torch.tensor(StandardScaler().fit_transform(X), dtype=torch.float32)
y_oh = torch.tensor(y_oh, dtype=torch.float32)

D, H = X.shape[1], 16
W1 = torch.randn(D, H) * 0.1
b1 = torch.zeros(H)
W2 = torch.randn(H, C) * 0.1
b2 = torch.zeros(C)

p = forward(X, W1, b1, W2, b2, out="softmax")
print("Iris CCE:", cce(p, y_oh).item())

Iris CCE: 1.0808532238006592


In [21]:
house = fetch_california_housing()
X = house.data
y = house.target.reshape(-1, 1)

X_train, _, y_train, _ = train_test_split(X, y, train_size=2000)

X = torch.tensor(StandardScaler().fit_transform(X_train), dtype=torch.float32)
y = torch.tensor(y_train, dtype=torch.float32)

D, H = X.shape[1], 32
W1 = torch.randn(D, H) * 0.1
b1 = torch.zeros(H)
W2 = torch.randn(H, 1) * 0.1
b2 = torch.zeros(1)

y_hat = forward(X, W1, b1, W2, b2, out="linear")
print("House MSE:", mse(y_hat, y).item())
print("House MAE:", mae(y_hat, y).item())

House MSE: 4.9240570068359375
House MAE: 1.9021700620651245
